# General Dataset

pip install torch torchvision --index-url https://download.pytorch.org/whl/cu126

In [38]:
import random
import gc
from typing import List, Dict, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

from datasets import load_dataset

from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Best hyperparameters (from your old code)
best_params = {
    'sentiment': {
        'learning_rate': 3.65445235521325e-05,
        'batch_size': 16,
        'num_epochs': 3,  # we’ll actually honor this now
        'warmup_ratio': 0.11560186404424366,
        'weight_decay': 0.02403950683025824,
        'hidden_dropout_prob': 0.1116167224336399,
        'classifier_dropout': 0.273235229154987
    },
    'emotion': {
        'learning_rate': 3.65445235521325e-05,
        'batch_size': 16,
        'num_epochs': 3,
        'warmup_ratio': 0.11560186404424366,
        'weight_decay': 0.02403950683025824,
        'hidden_dropout_prob': 0.1116167224336399,
        'classifier_dropout': 0.273235229154987
    }
}

# Seeds for analysis
SEEDS = [42, 123, 456, 789, 999]

Using device: cuda


In [39]:
def set_random_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [40]:
class DistilrobertaSingleTaskTransformer(nn.Module):
    def __init__(
        self,
        model_name: str = "distilroberta-base",
        num_classes: int = 3,
        hidden_dropout_prob: float = 0.1,
        attention_dropout_prob: float = 0.1,
        classifier_dropout: float = 0.1
    ):
        super().__init__()

        from transformers import AutoConfig
        config = AutoConfig.from_pretrained(model_name)
        config.hidden_dropout_prob = hidden_dropout_prob
        config.attention_probs_dropout_prob = attention_dropout_prob

        self.distilroberta = AutoModel.from_pretrained(model_name, config=config)
        self.dropout = nn.Dropout(classifier_dropout)
        self.classifier = nn.Linear(self.distilroberta.config.hidden_size, num_classes)

    def forward(self, input_ids, attention_mask):
        outputs = self.distilroberta(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = outputs.last_hidden_state[:, 0]  # first token
        pooled_output = self.dropout(pooled_output)
        logits = self.classifier(pooled_output)
        return {'logits': logits}


class DistilrobertaDataset(Dataset):
    def __init__(self, texts: List[str], labels: List[int], tokenizer, max_length: int = 128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = int(self.labels[idx])

        encoding = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(label, dtype=torch.long)
        }

In [41]:
def build_sst2_sentiment_dataset(
    split: str = "train",
    max_samples: int = None,
    neutral_prob: float = 0.15
) -> Tuple[List[str], List[int]]:
    """
    Build a 3-class sentiment dataset from SST-2:
    0 -> Negative (0)
    1 -> Neutral (1) with probability neutral_prob, otherwise Positive (2)
    """
    ds = load_dataset("sst2", split=split, num_proc=1)  # Disable parallelism
    texts = ds["sentence"]
    labels_raw = ds["label"]

    if max_samples is not None:
        texts = texts[:max_samples]
        labels_raw = labels_raw[:max_samples]

    labels_3 = []
    for lab in labels_raw:
        if lab == 0:
            labels_3.append(0)  # Negative
        else:  # lab == 1
            if np.random.random() < neutral_prob:
                labels_3.append(1)  # Neutral
            else:
                labels_3.append(2)  # Positive

    return texts, labels_3


def build_goemotions_emotion_dataset(
    split: str = "train",
    max_samples: int = None
) -> Tuple[List[str], List[int]]:
    """
    Build a 6-class emotion dataset from GoEmotions (simplified).
    Uses only labels in [0..5]. 
    If label is a list, uses the first element.
    """
    ds = load_dataset("go_emotions", "simplified", split=split, num_proc=1)  # Disable parallelism

    texts_all = ds["text"]
    labels_all = ds["labels"]

    texts = []
    labels = []

    for t, lab in zip(texts_all, labels_all):
        if isinstance(lab, list):
            if not lab:
                continue
            lab = lab[0]
        if 0 <= lab < 6:
            texts.append(t)
            labels.append(int(lab))

        if max_samples is not None and len(texts) >= max_samples:
            break

    return texts, labels

In [ ]:
import random
import gc
from typing import List, Dict, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

from datasets import load_dataset

from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Best hyperparameters (from your old code)
best_params = {
    'sentiment': {
        'learning_rate': 3.65445235521325e-05,
        'batch_size': 16,
        'num_epochs': 5,  # we’ll actually honor this now
        'warmup_ratio': 0.11560186404424366,
        'weight_decay': 0.02403950683025824,
        'hidden_dropout_prob': 0.1116167224336399,
        'classifier_dropout': 0.273235229154987
    },
    'emotion': {
        'learning_rate': 3.65445235521325e-05,
        'batch_size': 16,
        'num_epochs': 5,
        'warmup_ratio': 0.11560186404424366,
        'weight_decay': 0.02403950683025824,
        'hidden_dropout_prob': 0.1116167224336399,
        'classifier_dropout': 0.273235229154987
    }
}

# Seeds for analysis
SEEDS = [42, 123, 456, 789, 999]

def set_random_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

class DistilrobertaSingleTaskTransformer(nn.Module):
    def __init__(
        self,
        model_name: str = "distilroberta-base",
        num_classes: int = 3,
        hidden_dropout_prob: float = 0.1,
        attention_dropout_prob: float = 0.1,
        classifier_dropout: float = 0.1
    ):
        super().__init__()

        from transformers import AutoConfig
        config = AutoConfig.from_pretrained(model_name)
        config.hidden_dropout_prob = hidden_dropout_prob
        config.attention_probs_dropout_prob = attention_dropout_prob

        self.distilroberta = AutoModel.from_pretrained(model_name, config=config)
        self.dropout = nn.Dropout(classifier_dropout)
        self.classifier = nn.Linear(self.distilroberta.config.hidden_size, num_classes)

    def forward(self, input_ids, attention_mask):
        outputs = self.distilroberta(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = outputs.last_hidden_state[:, 0]  # first token
        pooled_output = self.dropout(pooled_output)
        logits = self.classifier(pooled_output)
        return {'logits': logits}


class DistilrobertaDataset(Dataset):
    def __init__(self, texts: List[str], labels: List[int], tokenizer, max_length: int = 128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = int(self.labels[idx])

        encoding = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(label, dtype=torch.long)
        }

def build_sst2_sentiment_dataset(
    split: str = "train",
    max_samples: int = None,
    neutral_prob: float = 0.15
) -> Tuple[List[str], List[int]]:
    """
    Build a 3-class sentiment dataset from SST-2:
    0 -> Negative (0)
    1 -> Neutral (1) with probability neutral_prob, otherwise Positive (2)
    """
    ds = load_dataset("sst2", split=split, num_proc=1)  # Disable parallelism
    texts = ds["sentence"]
    labels_raw = ds["label"]

    if max_samples is not None:
        texts = texts[:max_samples]
        labels_raw = labels_raw[:max_samples]

    labels_3 = []
    for lab in labels_raw:
        if lab == 0:
            labels_3.append(0)  # Negative
        else:  # lab == 1
            if np.random.random() < neutral_prob:
                labels_3.append(1)  # Neutral
            else:
                labels_3.append(2)  # Positive

    return texts, labels_3


def build_goemotions_emotion_dataset(
    split: str = "train",
    max_samples: int = None
) -> Tuple[List[str], List[int]]:
    """
    Build a 6-class emotion dataset from GoEmotions (simplified).
    Uses only labels in [0..5]. 
    If label is a list, uses the first element.
    """
    ds = load_dataset("go_emotions", "simplified", split=split, num_proc=1)  # Disable parallelism

    texts_all = ds["text"]
    labels_all = ds["labels"]

    texts = []
    labels = []

    for t, lab in zip(texts_all, labels_all):
        if isinstance(lab, list):
            if not lab:
                continue
            lab = lab[0]
        if 0 <= lab < 6:
            texts.append(t)
            labels.append(int(lab))

        if max_samples is not None and len(texts) >= max_samples:
            break

    return texts, labels

# You can cap max_samples if you want faster runs
MAX_SAMPLES_SENT = 3000  # or None for all
MAX_SAMPLES_EMOT = 3000  # or None for all

print("Building SST-2 sentiment dataset (3 classes)...")
sent_texts, sent_labels = build_sst2_sentiment_dataset(
    split="train",
    max_samples=MAX_SAMPLES_SENT,
    neutral_prob=0.15
)
print("Sentiment samples:", len(sent_texts))

print("\nBuilding GoEmotions emotion dataset (6 classes)...")
emot_texts, emot_labels = build_goemotions_emotion_dataset(
    split="train",
    max_samples=MAX_SAMPLES_EMOT
)
print("Emotion samples:", len(emot_texts))

Using device: cuda
Building SST-2 sentiment dataset (3 classes)...
Sentiment samples: 3000

Building GoEmotions emotion dataset (6 classes)...
Emotion samples: 3000


In [43]:
def cross_validate_single_task(
    task_type: str,
    texts: List[str],
    labels: List[int],
    best_params: Dict,
    seeds: List[int],
    num_folds: int = 5,
    max_length: int = 128
) -> Dict:
    """
    5-fold cross-validation across multiple seeds for single-task learning.
    Returns per-run metrics + overall mean/std for:
    - accuracy
    - macro F1
    - macro precision
    - macro recall
    """
    num_classes = 3 if task_type == "sentiment" else 6

    print(f"\n===== CROSS-VALIDATION for {task_type.capitalize()} =====")
    print(f"Samples: {len(texts)} | Folds: {num_folds} | Seeds: {seeds}")

    # Tokenizer shared across runs
    tokenizer = AutoTokenizer.from_pretrained("distilroberta-base")
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # Prepare folds once (same fold splits for all seeds)
    kf = KFold(n_splits=num_folds, shuffle=True, random_state=42)
    folds = list(kf.split(texts))  # Folds

    run_metrics = []

    for seed in seeds:
        print(f"\n🌱 Seed {seed}")
        set_random_seed(seed)

        accuracies = []
        f1s = []
        precisions = []
        recalls = []

        for fold_idx, (train_idx, val_idx) in enumerate(folds):
            print(f"  Fold {fold_idx + 1}/{num_folds}")

            # Split data for training and validation
            train_texts = [texts[i] for i in train_idx]
            train_labels = [labels[i] for i in train_idx]
            val_texts = [texts[i] for i in val_idx]
            val_labels = [labels[i] for i in val_idx]

            # Datasets & loaders
            train_dataset = DistilrobertaDataset(train_texts, train_labels, tokenizer, max_length)
            val_dataset = DistilrobertaDataset(val_texts, val_labels, tokenizer, max_length)

            train_loader = DataLoader(train_dataset, batch_size=best_params["batch_size"], shuffle=True)
            val_loader = DataLoader(val_dataset, batch_size=best_params["batch_size"], shuffle=False)

            # Model
            model = DistilrobertaSingleTaskTransformer(
                model_name="distilroberta-base",
                num_classes=num_classes,
                hidden_dropout_prob=best_params["hidden_dropout_prob"],
                attention_dropout_prob=best_params["hidden_dropout_prob"],
                classifier_dropout=best_params["classifier_dropout"]
            ).to(device)

            # Optimizer & scheduler
            optimizer = torch.optim.AdamW(model.parameters(), lr=best_params["learning_rate"], weight_decay=best_params["weight_decay"])

            num_epochs = best_params["num_epochs"]
            total_steps = len(train_loader) * num_epochs
            warmup_steps = int(total_steps * best_params["warmup_ratio"])
            scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps)

            criterion = nn.CrossEntropyLoss()

            # ----- Training ----- 
            model.train()
            for epoch in range(num_epochs):
                total_loss = 0.0
                for batch in train_loader:
                    optimizer.zero_grad()

                    input_ids = batch["input_ids"].to(device)
                    attention_mask = batch["attention_mask"].to(device)
                    labels_batch = batch["labels"].to(device)

                    outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                    loss = criterion(outputs["logits"], labels_batch)

                    loss.backward()
                    optimizer.step()
                    scheduler.step()

                avg_loss = total_loss / len(train_loader)
                print(f"    Epoch {epoch + 1}/{num_epochs} - Loss: {avg_loss:.4f}")

            # ----- Evaluation ----- 
            model.eval()
            all_preds = []
            all_true = []

            with torch.no_grad():
                for batch in val_loader:
                    input_ids = batch["input_ids"].to(device)
                    attention_mask = batch["attention_mask"].to(device)
                    labels_batch = batch["labels"].to(device)

                    outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                    preds = torch.argmax(outputs["logits"], dim=-1)

                    all_preds.extend(preds.cpu().numpy().tolist())
                    all_true.extend(labels_batch.cpu().numpy().tolist())

            # Calculate metrics
            acc = accuracy_score(all_true, all_preds)
            f1 = f1_score(all_true, all_preds, average="macro")
            precision = precision_score(all_true, all_preds, average="macro", zero_division=0)  # Handle zero-division warning
            recall = recall_score(all_true, all_preds, average="macro", zero_division=0)  # Handle zero-division warning

            accuracies.append(acc)
            f1s.append(f1)
            precisions.append(precision)
            recalls.append(recall)

        # Compute mean ± std for each metric
        def compute_mean_std(metrics):
            return f"{np.mean(metrics):.4f} ± {np.std(metrics):.4f}"

        print(f"Accuracy: {compute_mean_std(accuracies)}")
        print(f"Macro F1: {compute_mean_std(f1s)}")
        print(f"Macro Precision: {compute_mean_std(precisions)}")
        print(f"Macro Recall: {compute_mean_std(recalls)}")

    return run_metrics

# Now you can call the evaluation for sentiment and emotion tasks
sentiment_cv_results = cross_validate_single_task(
    task_type="sentiment",
    texts=sent_texts,
    labels=sent_labels,
    best_params=best_params["sentiment"],
    seeds=SEEDS,
    num_folds=5,
    max_length=128
)

emotion_cv_results = cross_validate_single_task(
    task_type="emotion",
    texts=emot_texts,
    labels=emot_labels,
    best_params=best_params["emotion"],
    seeds=SEEDS,
    num_folds=5,
    max_length=128
)



===== CROSS-VALIDATION for Sentiment =====
Samples: 3000 | Folds: 5 | Seeds: [42, 123, 456, 789, 999]

🌱 Seed 42
  Fold 1/5
    Epoch 1/5 - Loss: 0.0000
    Epoch 2/5 - Loss: 0.0000
    Epoch 3/5 - Loss: 0.0000
    Epoch 4/5 - Loss: 0.0000
    Epoch 5/5 - Loss: 0.0000
  Fold 2/5
    Epoch 1/5 - Loss: 0.0000
    Epoch 2/5 - Loss: 0.0000
    Epoch 3/5 - Loss: 0.0000
    Epoch 4/5 - Loss: 0.0000
    Epoch 5/5 - Loss: 0.0000
  Fold 3/5
    Epoch 1/5 - Loss: 0.0000
    Epoch 2/5 - Loss: 0.0000
    Epoch 3/5 - Loss: 0.0000
    Epoch 4/5 - Loss: 0.0000
    Epoch 5/5 - Loss: 0.0000
  Fold 4/5
    Epoch 1/5 - Loss: 0.0000
    Epoch 2/5 - Loss: 0.0000
    Epoch 3/5 - Loss: 0.0000
    Epoch 4/5 - Loss: 0.0000
    Epoch 5/5 - Loss: 0.0000
  Fold 5/5
    Epoch 1/5 - Loss: 0.0000
    Epoch 2/5 - Loss: 0.0000
    Epoch 3/5 - Loss: 0.0000
    Epoch 4/5 - Loss: 0.0000
    Epoch 5/5 - Loss: 0.0000
Accuracy: 0.8157 ± 0.0197
Macro F1: 0.5727 ± 0.0168
Macro Precision: 0.5619 ± 0.0392
Macro Recall: 0.5972 

In [44]:
# def build_sst2_sentiment_dataset(
#     split: str = "train",
#     max_samples: int = None,
#     neutral_prob: float = 0.15
# ) -> Tuple[List[str], List[int]]:
#     """
#     Build a 3-class sentiment dataset from SST-2:
#     0 -> Negative (0)
#     1 -> Neutral (1) with probability neutral_prob, otherwise Positive (2)
#     """
#     ds = load_dataset("sst2", split=split, num_proc=1)  # Disable parallelism
#     texts = ds["sentence"]
#     labels_raw = ds["label"]

#     if max_samples is not None:
#         texts = texts[:max_samples]
#         labels_raw = labels_raw[:max_samples]

#     labels_3 = []
#     for lab in labels_raw:
#         if lab == 0:
#             labels_3.append(0)  # Negative
#         else:  # lab == 1
#             if np.random.random() < neutral_prob:
#                 labels_3.append(1)  # Neutral
#             else:
#                 labels_3.append(2)  # Positive

#     return texts, labels_3


# def build_goemotions_emotion_dataset(
#     split: str = "train",
#     max_samples: int = None
# ) -> Tuple[List[str], List[int]]:
#     """
#     Build a 6-class emotion dataset from GoEmotions (simplified).
#     Uses only labels in [0..5]. 
#     If label is a list, uses the first element.
#     """
#     ds = load_dataset("go_emotions", "simplified", split=split, num_proc=1)  # Disable parallelism

#     texts_all = ds["text"]
#     labels_all = ds["labels"]

#     texts = []
#     labels = []

#     for t, lab in zip(texts_all, labels_all):
#         if isinstance(lab, list):
#             if not lab:
#                 continue
#             lab = lab[0]
#         if 0 <= lab < 6:
#             texts.append(t)
#             labels.append(int(lab))

#         if max_samples is not None and len(texts) >= max_samples:
#             break

#     return texts, labels

#### Multitask

In [45]:
import random
import gc
from typing import List, Dict, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

from datasets import load_dataset

from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Best hyperparameters for multi-task model
best_params = {
    'learning_rate': 6.251028636335231e-05,
    'batch_size': 32,
    'num_epochs': 6,
    'warmup_ratio': 0.12123391106782762,
    'weight_decay': 0.02636424704863906,
    'hidden_dropout_prob': 0.13668090197068677,
    'classifier_dropout': 0.16084844859190756,
    'alpha': 0.5049512863264476
}

# Seeds for analysis
SEEDS = [42, 123, 456, 789, 999]

def set_random_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

class DistilrobertaMultiTaskTransformer(nn.Module):
    def __init__(
        self,
        model_name: str = "distilroberta-base",
        sentiment_num_classes: int = 3,
        emotion_num_classes: int = 6,
        hidden_dropout_prob: float = 0.1,
        attention_dropout_prob: float = 0.1,
        classifier_dropout: float = 0.1
    ):
        super().__init__()

        from transformers import AutoConfig
        config = AutoConfig.from_pretrained(model_name)
        config.hidden_dropout_prob = hidden_dropout_prob
        config.attention_probs_dropout_prob = attention_dropout_prob

        self.distilroberta = AutoModel.from_pretrained(model_name, config=config)
        self.dropout = nn.Dropout(classifier_dropout)
        
        # Classifier heads for sentiment and emotion tasks
        self.sentiment_classifier = nn.Linear(self.distilroberta.config.hidden_size, sentiment_num_classes)
        self.emotion_classifier = nn.Linear(self.distilroberta.config.hidden_size, emotion_num_classes)

    def forward(self, input_ids, attention_mask):
        outputs = self.distilroberta(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = outputs.last_hidden_state[:, 0]  # first token
        pooled_output = self.dropout(pooled_output)

        # Compute logits for both tasks
        sentiment_logits = self.sentiment_classifier(pooled_output)
        emotion_logits = self.emotion_classifier(pooled_output)
        
        return {'sentiment_logits': sentiment_logits, 'emotion_logits': emotion_logits}


class DistilrobertaMultiTaskDataset(Dataset):
    def __init__(self, texts: List[str], sentiment_labels: List[int], emotion_labels: List[int], tokenizer, max_length: int = 128):
        self.texts = texts
        self.sentiment_labels = sentiment_labels
        self.emotion_labels = emotion_labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        sentiment_label = int(self.sentiment_labels[idx])
        emotion_label = int(self.emotion_labels[idx])

        encoding = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "sentiment_labels": torch.tensor(sentiment_label, dtype=torch.long),
            "emotion_labels": torch.tensor(emotion_label, dtype=torch.long)
        }

def cross_validate_distilroberta_multi_task(
    sentiment_texts: List[str],
    sentiment_labels: List[int],
    emotion_texts: List[str],
    emotion_labels: List[int],
    best_params: Dict,
    seeds: List[int],
    num_folds: int = 5,
    max_length: int = 128
) -> Dict:
    """
    5-fold cross-validation across multiple seeds for multi-task learning.
    Returns per-run metrics + overall mean/std for:
    - accuracy
    - macro F1
    - macro precision
    - macro recall
    """
    sentiment_num_classes = 3
    emotion_num_classes = 6

    print(f"\n===== CROSS-VALIDATION for MULTI-TASK =====")
    print(f"Sentiment Samples: {len(sentiment_texts)} | Emotion Samples: {len(emotion_texts)} | Folds: {num_folds} | Seeds: {seeds}")

    # Tokenizer shared across runs
    tokenizer = AutoTokenizer.from_pretrained("distilroberta-base")
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # Prepare folds once (same fold splits for all seeds)
    kf = KFold(n_splits=num_folds, shuffle=True, random_state=42)
    sentiment_folds = list(kf.split(sentiment_texts))  # Sentiment folds
    emotion_folds = list(kf.split(emotion_texts))  # Emotion folds

    run_metrics = []

    for seed in seeds:
        print(f"\n🌱 Seed {seed}")
        set_random_seed(seed)

        sentiment_accuracies = []
        emotion_accuracies = []
        sentiment_f1s = []
        emotion_f1s = []
        sentiment_precisions = []
        emotion_precisions = []
        sentiment_recalls = []
        emotion_recalls = []

        for fold_idx, (sentiment_train_idx, sentiment_val_idx) in enumerate(sentiment_folds):
            print(f"  Fold {fold_idx + 1}/{num_folds}")

            # Handle both sentiment and emotion datasets in the same fold
            sentiment_train_texts = [sentiment_texts[i] for i in sentiment_train_idx]
            sentiment_train_labels = [sentiment_labels[i] for i in sentiment_train_idx]
            sentiment_val_texts = [sentiment_texts[i] for i in sentiment_val_idx]
            sentiment_val_labels = [sentiment_labels[i] for i in sentiment_val_idx]

            # Emotion data split for the same fold
            emotion_train_texts = [emotion_texts[i] for i in sentiment_train_idx]
            emotion_train_labels = [emotion_labels[i] for i in sentiment_train_idx]
            emotion_val_texts = [emotion_texts[i] for i in sentiment_val_idx]
            emotion_val_labels = [emotion_labels[i] for i in sentiment_val_idx]

            # Datasets & loaders
            train_dataset = DistilrobertaMultiTaskDataset(
                sentiment_train_texts, sentiment_train_labels, emotion_train_labels, tokenizer, max_length
            )
            val_dataset = DistilrobertaMultiTaskDataset(
                sentiment_val_texts, sentiment_val_labels, emotion_val_labels, tokenizer, max_length
            )

            train_loader = DataLoader(train_dataset, batch_size=best_params["batch_size"], shuffle=True)
            val_loader = DataLoader(val_dataset, batch_size=best_params["batch_size"], shuffle=False)

            # Model
            model = DistilrobertaMultiTaskTransformer(
                model_name="distilroberta-base",
                sentiment_num_classes=sentiment_num_classes,
                emotion_num_classes=emotion_num_classes,
                hidden_dropout_prob=best_params["hidden_dropout_prob"],
                attention_dropout_prob=best_params["hidden_dropout_prob"],
                classifier_dropout=best_params["classifier_dropout"]
            ).to(device)

            # Optimizer & scheduler
            optimizer = torch.optim.AdamW(model.parameters(), lr=best_params["learning_rate"], weight_decay=best_params["weight_decay"])

            num_epochs = best_params.get("num_epochs", 6)
            total_steps = len(train_loader) * num_epochs
            warmup_steps = int(total_steps * best_params["warmup_ratio"])

            scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps)

            criterion = nn.CrossEntropyLoss()

            # ----- Training ----- 
            model.train()
            for epoch in range(num_epochs):
                total_loss = 0.0
                for batch in train_loader:
                    optimizer.zero_grad()

                    input_ids = batch["input_ids"].to(device)
                    attention_mask = batch["attention_mask"].to(device)
                    sentiment_labels_batch = batch["sentiment_labels"].to(device)
                    emotion_labels_batch = batch["emotion_labels"].to(device)

                    outputs = model(input_ids=input_ids, attention_mask=attention_mask)

                    sentiment_loss = criterion(outputs["sentiment_logits"], sentiment_labels_batch)
                    emotion_loss = criterion(outputs["emotion_logits"], emotion_labels_batch)

                    total_loss = sentiment_loss + best_params["alpha"] * emotion_loss

                    total_loss.backward()
                    optimizer.step()
                    scheduler.step()

                avg_loss = total_loss / max(1, len(train_loader))
                print(f"    Epoch {epoch + 1}/{num_epochs} - Loss: {avg_loss:.4f}")

            # ----- Evaluation ----- 
            model.eval()
            all_sentiment_preds = []
            all_emotion_preds = []
            all_sentiment_true = []
            all_emotion_true = []

            with torch.no_grad():
                for batch in val_loader:
                    input_ids = batch["input_ids"].to(device)
                    attention_mask = batch["attention_mask"].to(device)
                    sentiment_labels_batch = batch["sentiment_labels"].to(device)
                    emotion_labels_batch = batch["emotion_labels"].to(device)

                    outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                    sentiment_logits = outputs["sentiment_logits"]
                    emotion_logits = outputs["emotion_logits"]

                    sentiment_preds = torch.argmax(sentiment_logits, dim=-1)
                    emotion_preds = torch.argmax(emotion_logits, dim=-1)

                    # Separate predictions and labels for sentiment and emotion
                    all_sentiment_preds.extend(sentiment_preds.cpu().numpy().tolist())
                    all_emotion_preds.extend(emotion_preds.cpu().numpy().tolist())
                    all_sentiment_true.extend(sentiment_labels_batch.cpu().numpy().tolist())
                    all_emotion_true.extend(emotion_labels_batch.cpu().numpy().tolist())

            # Sentiment task metrics
            sentiment_acc = accuracy_score(all_sentiment_true, all_sentiment_preds)
            sentiment_macro_f1 = f1_score(all_sentiment_true, all_sentiment_preds, average="macro", zero_division=0)
            sentiment_macro_prec = precision_score(all_sentiment_true, all_sentiment_preds, average="macro", zero_division=0)
            sentiment_macro_rec = recall_score(all_sentiment_true, all_sentiment_preds, average="macro", zero_division=0)

            # Emotion task metrics
            emotion_acc = accuracy_score(all_emotion_true, all_emotion_preds)
            emotion_macro_f1 = f1_score(all_emotion_true, all_emotion_preds, average="macro", zero_division=0)
            emotion_macro_prec = precision_score(all_emotion_true, all_emotion_preds, average="macro", zero_division=0)
            emotion_macro_rec = recall_score(all_emotion_true, all_emotion_preds, average="macro", zero_division=0)

            sentiment_accuracies.append(sentiment_acc)
            emotion_accuracies.append(emotion_acc)
            sentiment_f1s.append(sentiment_macro_f1)
            emotion_f1s.append(emotion_macro_f1)
            sentiment_precisions.append(sentiment_macro_prec)
            emotion_precisions.append(emotion_macro_prec)
            sentiment_recalls.append(sentiment_macro_rec)
            emotion_recalls.append(emotion_macro_rec)

        # Calculate mean ± std for each metric
        def compute_mean_std(metrics):
            return f"{np.mean(metrics):.4f} ± {np.std(metrics):.4f}"

        print(f"Sentiment Accuracy: {compute_mean_std(sentiment_accuracies)}")
        print(f"Emotion Accuracy: {compute_mean_std(emotion_accuracies)}")
        print(f"Sentiment Macro F1: {compute_mean_std(sentiment_f1s)}")
        print(f"Emotion Macro F1: {compute_mean_std(emotion_f1s)}")
        print(f"Sentiment Macro Precision: {compute_mean_std(sentiment_precisions)}")
        print(f"Emotion Macro Precision: {compute_mean_std(emotion_precisions)}")
        print(f"Sentiment Macro Recall: {compute_mean_std(sentiment_recalls)}")
        print(f"Emotion Macro Recall: {compute_mean_std(emotion_recalls)}")

    return run_metrics

# Load datasets for sentiment and emotion tasks
MAX_SAMPLES_SENT = 3000
MAX_SAMPLES_EMOT = 3000

print("Building SST-2 sentiment dataset (3 classes)...")
sent_texts, sent_labels = build_sst2_sentiment_dataset(split="train", max_samples=MAX_SAMPLES_SENT, neutral_prob=0.15)
print("Sentiment samples:", len(sent_texts))

print("\nBuilding GoEmotions emotion dataset (6 classes)...")
emot_texts, emot_labels = build_goemotions_emotion_dataset(split="train", max_samples=MAX_SAMPLES_EMOT)
print("Emotion samples:", len(emot_texts))

# 5-fold CV for multi-task model
multi_task_cv_results = cross_validate_distilroberta_multi_task(
    sentiment_texts=sent_texts,
    sentiment_labels=sent_labels,
    emotion_texts=emot_texts,
    emotion_labels=emot_labels,
    best_params=best_params,
    seeds=SEEDS,
    num_folds=5,
    max_length=128
)

Using device: cuda
Building SST-2 sentiment dataset (3 classes)...
Sentiment samples: 3000

Building GoEmotions emotion dataset (6 classes)...
Emotion samples: 3000

===== CROSS-VALIDATION for MULTI-TASK =====
Sentiment Samples: 3000 | Emotion Samples: 3000 | Folds: 5 | Seeds: [42, 123, 456, 789, 999]

🌱 Seed 42
  Fold 1/5
    Epoch 1/6 - Loss: 0.0206
    Epoch 2/6 - Loss: 0.0165
    Epoch 3/6 - Loss: 0.0207
    Epoch 4/6 - Loss: 0.0128
    Epoch 5/6 - Loss: 0.0157
    Epoch 6/6 - Loss: 0.0130
  Fold 2/5
    Epoch 1/6 - Loss: 0.0166
    Epoch 2/6 - Loss: 0.0217
    Epoch 3/6 - Loss: 0.0140
    Epoch 4/6 - Loss: 0.0160
    Epoch 5/6 - Loss: 0.0156
    Epoch 6/6 - Loss: 0.0131
  Fold 3/5
    Epoch 1/6 - Loss: 0.0223
    Epoch 2/6 - Loss: 0.0188
    Epoch 3/6 - Loss: 0.0268
    Epoch 4/6 - Loss: 0.0161
    Epoch 5/6 - Loss: 0.0173
    Epoch 6/6 - Loss: 0.0142
  Fold 4/5
    Epoch 1/6 - Loss: 0.0201
    Epoch 2/6 - Loss: 0.0216
    Epoch 3/6 - Loss: 0.0156
    Epoch 4/6 - Loss: 0.0170
    

# Reddit specific dataset

In [46]:
import random
import gc
import pandas as pd
from typing import List, Dict, Tuple
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Best hyperparameters (for sentiment, emotion, and multitask)
best_params = {
    'sentiment': {
        'learning_rate': 3.65445235521325e-05,
        'batch_size': 16,
        'num_epochs': 5,  # we'll actually honor this now
        'warmup_ratio': 0.11560186404424366,
        'weight_decay': 0.02403950683025824,
        'hidden_dropout_prob': 0.1116167224336399,
        'classifier_dropout': 0.273235229154987
    },
    'emotion': {
        'learning_rate': 3.65445235521325e-05,
        'batch_size': 16,
        'num_epochs': 5,
        'warmup_ratio': 0.11560186404424366,
        'weight_decay': 0.02403950683025824,
        'hidden_dropout_prob': 0.1116167224336399,
        'classifier_dropout': 0.273235229154987
    },
    'multitask': {
        'learning_rate': 6.251028636335231e-05,
        'batch_size': 32,
        'num_epochs': 6,
        'warmup_ratio': 0.12123391106782762,
        'weight_decay': 0.02636424704863906,
        'hidden_dropout_prob': 0.13668090197068677,
        'classifier_dropout': 0.16084844859190756,
        'alpha': 0.5049512863264476,
    }}

# Seeds for analysis
SEEDS = [42, 123, 456, 789, 999]

Using device: cuda


In [47]:
def set_random_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

class DistilrobertaSingleTaskTransformer(nn.Module):
    def __init__(self, model_name="distilroberta-base", num_classes=3, hidden_dropout_prob=0.1, attention_dropout_prob=0.1, classifier_dropout=0.1):
        super().__init__()
        from transformers import AutoConfig
        config = AutoConfig.from_pretrained(model_name)
        config.hidden_dropout_prob = hidden_dropout_prob
        config.attention_probs_dropout_prob = attention_dropout_prob

        self.distilroberta = AutoModel.from_pretrained(model_name, config=config)
        self.dropout = nn.Dropout(classifier_dropout)
        self.classifier = nn.Linear(self.distilroberta.config.hidden_size, num_classes)

    def forward(self, input_ids, attention_mask):
        outputs = self.distilroberta(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = outputs.last_hidden_state[:, 0]  # first token
        pooled_output = self.dropout(pooled_output)
        logits = self.classifier(pooled_output)
        return {'logits': logits}

class DistilrobertaDataset(Dataset):
    def __init__(self, texts: List[str], labels: List[int], tokenizer, max_length: int = 128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = int(self.labels[idx])

        encoding = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(label, dtype=torch.long)
        }

In [48]:
import pandas as pd

def load_reddit_dataset(csv_path: str):
    df = pd.read_csv(csv_path)
    
    # Check the dataset to confirm the columns
    print(df.columns)
    
    # Sentiment Label Mapping
    sentiment_label_map = {
        'Negative': 0,
        'Neutral': 1,
        'Positive': 2
    }
    
    # Emotion Label Mapping
    emotion_label_map = {
        'Sadness': 0,
        'Surprise': 1,
        'Anger': 2,
        'No Emotion': 3,  # Treating 'No Emotion' as a separate class
        'Joy': 4,
        'Fear': 5
    }

    # Convert sentiment and emotion labels to integers
    df['sentiment'] = df['sentiment'].map(sentiment_label_map)
    df['emotion'] = df['emotion'].map(emotion_label_map)

    # Check if mapping was successful
    print("Mapped sentiment labels:", df['sentiment'].unique())
    print("Mapped emotion labels:", df['emotion'].unique())

    # Return dataset
    texts = df['text_content'].tolist()  # Assuming 'test_content' is the text column
    sentiment_labels = df['sentiment'].tolist()
    emotion_labels = df['emotion'].tolist()

    return {
        "texts": texts,
        "sentiment_labels": sentiment_labels,
        "emotion_labels": emotion_labels
    }

# Path to your annotated Reddit CSV
REDDIT_CSV_PATH = "annotated_reddit_posts.csv"  # change if needed

# Load the dataset
reddit_data = load_reddit_dataset(REDDIT_CSV_PATH)
texts = reddit_data["texts"]
sent_labels = reddit_data["sentiment_labels"]
emo_labels = reddit_data["emotion_labels"]

Index(['id', 'text_content', 'original_text', 'type', 'score', 'subjectivity',
       'sentiment', 'emotion'],
      dtype='object')
Mapped sentiment labels: [0 1 2]
Mapped emotion labels: [0 1 2 3 4 5]


In [49]:
def cross_validate_distilroberta_single_task(
    task_type: str,
    texts: List[str],
    labels: List[int],
    best_params_task: Dict,
    seeds: List[int],
    num_folds: int = 5,
    max_length: int = 128
) -> Dict:
    """
    5-fold cross-validation across multiple seeds for single-task learning.
    Returns per-run metrics + overall mean/std for:
    - accuracy
    - macro F1
    - macro precision
    - macro recall
    """
    num_classes = 3 if task_type == "sentiment" else 6
    print(f"\n===== CROSS-VALIDATION for {task_type.capitalize()} =====")
    print(f"Samples: {len(texts)} | Folds: {num_folds} | Seeds: {seeds}")

    # Tokenizer shared across runs
    tokenizer = AutoTokenizer.from_pretrained("distilroberta-base")
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # Prepare folds once (same fold splits for all seeds)
    kf = KFold(n_splits=num_folds, shuffle=True, random_state=42)
    folds = list(kf.split(texts))  # Folds

    run_metrics = []

    for seed in seeds:
        print(f"\n🌱 Seed {seed}")
        set_random_seed(seed)

        accuracies = []
        f1s = []
        precisions = []
        recalls = []

        for fold_idx, (train_idx, val_idx) in enumerate(folds):
            print(f"  Fold {fold_idx + 1}/{num_folds}")

            # Split data for training and validation
            train_texts = [texts[i] for i in train_idx]
            train_labels = [labels[i] for i in train_idx]
            val_texts = [texts[i] for i in val_idx]
            val_labels = [labels[i] for i in val_idx]

            # Datasets & loaders
            train_dataset = DistilrobertaDataset(train_texts, train_labels, tokenizer, max_length)
            val_dataset = DistilrobertaDataset(val_texts, val_labels, tokenizer, max_length)

            train_loader = DataLoader(train_dataset, batch_size=best_params_task["batch_size"], shuffle=True)
            val_loader = DataLoader(val_dataset, batch_size=best_params_task["batch_size"], shuffle=False)

            # Model
            model = DistilrobertaSingleTaskTransformer(
                model_name="distilroberta-base",
                num_classes=num_classes,
                hidden_dropout_prob=best_params_task["hidden_dropout_prob"],
                attention_dropout_prob=best_params_task["hidden_dropout_prob"],
                classifier_dropout=best_params_task["classifier_dropout"]
            ).to(device)

            # Optimizer & scheduler
            optimizer = torch.optim.AdamW(model.parameters(), lr=best_params_task["learning_rate"], weight_decay=best_params_task["weight_decay"])

            num_epochs = best_params_task["num_epochs"]
            total_steps = len(train_loader) * num_epochs
            warmup_steps = int(total_steps * best_params_task["warmup_ratio"])
            scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps)

            criterion = nn.CrossEntropyLoss()

            # ----- Training ----- 
            model.train()
            for epoch in range(num_epochs):
                total_loss = 0.0
                for batch in train_loader:
                    optimizer.zero_grad()

                    input_ids = batch["input_ids"].to(device)
                    attention_mask = batch["attention_mask"].to(device)
                    labels_batch = batch["labels"].to(device)

                    outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                    loss = criterion(outputs["logits"], labels_batch)

                    loss.backward()
                    optimizer.step()
                    scheduler.step()

                avg_loss = total_loss / len(train_loader)
                print(f"    Epoch {epoch + 1}/{num_epochs} - Loss: {avg_loss:.4f}")

            # ----- Evaluation ----- 
            model.eval()
            all_preds = []
            all_true = []

            with torch.no_grad():
                for batch in val_loader:
                    input_ids = batch["input_ids"].to(device)
                    attention_mask = batch["attention_mask"].to(device)
                    labels_batch = batch["labels"].to(device)

                    outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                    preds = torch.argmax(outputs["logits"], dim=-1)

                    all_preds.extend(preds.cpu().numpy().tolist())
                    all_true.extend(labels_batch.cpu().numpy().tolist())

            # Calculate metrics
            acc = accuracy_score(all_true, all_preds)
            f1 = f1_score(all_true, all_preds, average="macro")
            precision = precision_score(all_true, all_preds, average="macro", zero_division=0)
            recall = recall_score(all_true, all_preds, average="macro", zero_division=0)

            accuracies.append(acc)
            f1s.append(f1)
            precisions.append(precision)
            recalls.append(recall)

        # Compute mean ± std for each metric
        def compute_mean_std(metrics):
            return f"{np.mean(metrics):.4f} ± {np.std(metrics):.4f}"

        print(f"Accuracy: {compute_mean_std(accuracies)}")
        print(f"Macro F1: {compute_mean_std(f1s)}")
        print(f"Macro Precision: {compute_mean_std(precisions)}")
        print(f"Macro Recall: {compute_mean_std(recalls)}")

    return run_metrics

In [50]:
# Single-task sentiment evaluation
sentiment_cv_results = cross_validate_distilroberta_single_task(
    task_type="sentiment",  # Sentiment task
    texts=texts,  # Reddit post texts
    labels=sent_labels,  # Sentiment labels
    best_params_task=best_params["sentiment"],  # Best parameters for sentiment
    seeds=SEEDS,  # Seeds for randomness
    num_folds=5,  # Cross-validation folds
    max_length=128  # Max sequence length for the model
)

# Single-task emotion evaluation
emotion_cv_results = cross_validate_distilroberta_single_task(
    task_type="emotion",  # Emotion task
    texts=texts,  # Reddit post texts
    labels=emo_labels,  # Emotion labels
    best_params_task=best_params["emotion"],  # Best parameters for emotion
    seeds=SEEDS,  # Seeds for randomness
    num_folds=5,  # Cross-validation folds
    max_length=128  # Max sequence length for the model
)


===== CROSS-VALIDATION for Sentiment =====
Samples: 95 | Folds: 5 | Seeds: [42, 123, 456, 789, 999]

🌱 Seed 42
  Fold 1/5
    Epoch 1/5 - Loss: 0.0000
    Epoch 2/5 - Loss: 0.0000
    Epoch 3/5 - Loss: 0.0000
    Epoch 4/5 - Loss: 0.0000
    Epoch 5/5 - Loss: 0.0000
  Fold 2/5
    Epoch 1/5 - Loss: 0.0000
    Epoch 2/5 - Loss: 0.0000
    Epoch 3/5 - Loss: 0.0000
    Epoch 4/5 - Loss: 0.0000
    Epoch 5/5 - Loss: 0.0000
  Fold 3/5
    Epoch 1/5 - Loss: 0.0000
    Epoch 2/5 - Loss: 0.0000
    Epoch 3/5 - Loss: 0.0000
    Epoch 4/5 - Loss: 0.0000
    Epoch 5/5 - Loss: 0.0000
  Fold 4/5
    Epoch 1/5 - Loss: 0.0000
    Epoch 2/5 - Loss: 0.0000
    Epoch 3/5 - Loss: 0.0000
    Epoch 4/5 - Loss: 0.0000
    Epoch 5/5 - Loss: 0.0000
  Fold 5/5
    Epoch 1/5 - Loss: 0.0000
    Epoch 2/5 - Loss: 0.0000
    Epoch 3/5 - Loss: 0.0000
    Epoch 4/5 - Loss: 0.0000
    Epoch 5/5 - Loss: 0.0000
Accuracy: 0.5474 ± 0.1272
Macro F1: 0.2326 ± 0.0388
Macro Precision: 0.1825 ± 0.0424
Macro Recall: 0.3333 ± 

In [51]:
##Multitask

In [52]:
import random
import gc
import pandas as pd
from typing import List, Dict, Tuple
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

# Set device to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Load Reddit Dataset
def load_reddit_dataset(csv_path: str):
    df = pd.read_csv(csv_path)
    print(df.columns)  # Checking columns to debug the dataset
    texts = df['text_content'].tolist()  # Assuming 'text_content' column contains the posts
    sentiment_labels = df['sentiment'].tolist()  # Assuming 'sentiment' column contains sentiment labels
    emotion_labels = df['emotion'].tolist()  # Assuming 'emotion' column contains emotion labels
    
    # Map string labels to integers
    sentiment_map = {'Negative': 0, 'Neutral': 1, 'Positive': 2}
    emotion_map = {'Sadness': 0, 'Surprise': 1, 'Anger': 2, 'No Emotion': 3, 'Joy': 4, 'Fear': 5}
    
    sentiment_labels = [sentiment_map[label] for label in sentiment_labels]
    emotion_labels = [emotion_map[label] for label in emotion_labels]
    
    return {"texts": texts, "sentiment_labels": sentiment_labels, "emotion_labels": emotion_labels}

# Load your Reddit CSV file
REDDIT_CSV_PATH = "annotated_reddit_posts.csv"  # Change if needed
reddit_data = load_reddit_dataset(REDDIT_CSV_PATH)

texts = reddit_data["texts"]
sent_labels = reddit_data["sentiment_labels"]
emo_labels = reddit_data["emotion_labels"]

# Hyperparameters
best_params = {
    'sentiment': {
        'learning_rate': 3.65445235521325e-05,
        'batch_size': 16,
        'num_epochs': 3,
        'warmup_ratio': 0.11560186404424366,
        'weight_decay': 0.02403950683025824,
        'hidden_dropout_prob': 0.1116167224336399,
        'classifier_dropout': 0.273235229154987
    },
    'emotion': {
        'learning_rate': 3.65445235521325e-05,
        'batch_size': 16,
        'num_epochs': 3,
        'warmup_ratio': 0.11560186404424366,
        'weight_decay': 0.02403950683025824,
        'hidden_dropout_prob': 0.1116167224336399,
        'classifier_dropout': 0.273235229154987
    },
    'multitask': {
        'learning_rate': 6.251028636335231e-05,
        'batch_size': 32,
        'num_epochs': 6,
        'warmup_ratio': 0.12123391106782762,
        'weight_decay': 0.02636424704863906,
        'hidden_dropout_prob': 0.13668090197068677,
        'classifier_dropout': 0.16084844859190756,
        'alpha': 0.5049512863264476,
    }
}

# Set random seed for reproducibility
def set_random_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

# Clear memory to avoid memory issues
def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# Dataset Class for Multi-task (Sentiment + Emotion)
class DistilrobertaMultiTaskDataset(Dataset):
    def __init__(self, texts: List[str], sentiment_labels: List[int], emotion_labels: List[int], tokenizer, max_length: int = 128):
        self.texts = texts
        self.sentiment_labels = sentiment_labels
        self.emotion_labels = emotion_labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        sentiment_label = int(self.sentiment_labels[idx])
        emotion_label = int(self.emotion_labels[idx])

        encoding = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "sentiment_labels": torch.tensor(sentiment_label, dtype=torch.long),
            "emotion_labels": torch.tensor(emotion_label, dtype=torch.long)
        }

# Multi-task Model: Sentiment + Emotion
class DistilrobertaMultiTaskTransformer(nn.Module):
    def __init__(self, model_name="distilroberta-base", sentiment_num_classes=3, emotion_num_classes=6):
        super().__init__()
        from transformers import AutoConfig, AutoModel

        config = AutoConfig.from_pretrained(model_name)
        self.distilroberta = AutoModel.from_pretrained(model_name, config=config)
        self.sentiment_classifier = nn.Linear(config.hidden_size, sentiment_num_classes)
        self.emotion_classifier = nn.Linear(config.hidden_size, emotion_num_classes)

    def forward(self, input_ids, attention_mask):
        outputs = self.distilroberta(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = outputs.last_hidden_state[:, 0]  # First token
        sentiment_logits = self.sentiment_classifier(pooled_output)
        emotion_logits = self.emotion_classifier(pooled_output)
        return {'sentiment_logits': sentiment_logits, 'emotion_logits': emotion_logits}

# Cross-validation for Multi-task model
def cross_validate_distilroberta_multitask(
    texts: List[str], 
    sentiment_labels: List[int], 
    emotion_labels: List[int], 
    best_params_task: Dict, 
    seeds: List[int], 
    num_folds: int = 5, 
    max_length: int = 128
) -> Dict:
    sentiment_num_classes = 3
    emotion_num_classes = 6

    print(f"\n===== CROSS-VALIDATION for MULTI-TASK =====")
    print(f"Sentiment Samples: {len(sentiment_labels)} | Emotion Samples: {len(emotion_labels)} | Folds: {num_folds} | Seeds: {seeds}")

    tokenizer = AutoTokenizer.from_pretrained("distilroberta-base")
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # Prepare the folds
    kf = KFold(n_splits=num_folds, shuffle=True, random_state=42)
    sentiment_folds = list(kf.split(sentiment_labels))
    emotion_folds = list(kf.split(emotion_labels))

    run_metrics = []

    for seed in seeds:
        print(f"\n🌱 Seed {seed}")
        set_random_seed(seed)

        # Track metrics for the seed
        seed_metrics = {
            "sentiment_accuracy": [],
            "emotion_accuracy": [],
            "sentiment_macro_f1": [],
            "emotion_macro_f1": [],
            "sentiment_macro_precision": [],
            "emotion_macro_precision": [],
            "sentiment_macro_recall": [],
            "emotion_macro_recall": [],
        }

        for fold_idx, (sentiment_train_idx, sentiment_val_idx) in enumerate(sentiment_folds):
            print(f"  Fold {fold_idx + 1}/{num_folds}")

            sentiment_train_texts = [texts[i] for i in sentiment_train_idx]
            sentiment_train_labels = [sentiment_labels[i] for i in sentiment_train_idx]
            sentiment_val_texts = [texts[i] for i in sentiment_val_idx]
            sentiment_val_labels = [sentiment_labels[i] for i in sentiment_val_idx]

            emotion_train_texts = [texts[i] for i in sentiment_train_idx]
            emotion_train_labels = [emotion_labels[i] for i in sentiment_train_idx]
            emotion_val_texts = [texts[i] for i in sentiment_val_idx]
            emotion_val_labels = [emotion_labels[i] for i in sentiment_val_idx]

            train_dataset = DistilrobertaMultiTaskDataset(
                sentiment_train_texts, sentiment_train_labels, emotion_train_labels, tokenizer, max_length
            )
            val_dataset = DistilrobertaMultiTaskDataset(
                sentiment_val_texts, sentiment_val_labels, emotion_val_labels, tokenizer, max_length
            )

            train_loader = DataLoader(train_dataset, batch_size=best_params_task["batch_size"], shuffle=True)
            val_loader = DataLoader(val_dataset, batch_size=best_params_task["batch_size"], shuffle=False)

            model = DistilrobertaMultiTaskTransformer(
                model_name="distilroberta-base",
                sentiment_num_classes=sentiment_num_classes,
                emotion_num_classes=emotion_num_classes
            ).to(device)

            optimizer = torch.optim.AdamW(model.parameters(), lr=best_params_task["learning_rate"], weight_decay=best_params_task["weight_decay"])
            num_epochs = best_params_task["num_epochs"]
            total_steps = len(train_loader) * num_epochs
            warmup_steps = int(total_steps * best_params_task["warmup_ratio"])

            scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps)
            sentiment_criterion = nn.CrossEntropyLoss()
            emotion_criterion = nn.CrossEntropyLoss()

            for epoch in range(num_epochs):
                model.train()
                total_loss = 0.0
                for batch in train_loader:
                    optimizer.zero_grad()

                    input_ids = batch["input_ids"].to(device)
                    attention_mask = batch["attention_mask"].to(device)
                    sentiment_labels_batch = batch["sentiment_labels"].to(device)
                    emotion_labels_batch = batch["emotion_labels"].to(device)

                    outputs = model(input_ids=input_ids, attention_mask=attention_mask)

                    sentiment_loss = sentiment_criterion(outputs["sentiment_logits"], sentiment_labels_batch)
                    emotion_loss = emotion_criterion(outputs["emotion_logits"], emotion_labels_batch)

                    total_loss = sentiment_loss + best_params_task["alpha"] * emotion_loss

                    total_loss.backward()
                    optimizer.step()
                    scheduler.step()

                avg_loss = total_loss / len(train_loader)
                print(f"    Epoch {epoch + 1}/{num_epochs} - Loss: {avg_loss:.4f}")

            # Evaluation
            model.eval()
            all_sentiment_preds = []
            all_emotion_preds = []
            all_sentiment_true = []
            all_emotion_true = []

            with torch.no_grad():
                for batch in val_loader:
                    input_ids = batch["input_ids"].to(device)
                    attention_mask = batch["attention_mask"].to(device)
                    sentiment_labels_batch = batch["sentiment_labels"].to(device)
                    emotion_labels_batch = batch["emotion_labels"].to(device)

                    outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                    sentiment_preds = torch.argmax(outputs["sentiment_logits"], dim=-1)
                    emotion_preds = torch.argmax(outputs["emotion_logits"], dim=-1)

                    all_sentiment_preds.extend(sentiment_preds.cpu().numpy().tolist())
                    all_emotion_preds.extend(emotion_preds.cpu().numpy().tolist())
                    all_sentiment_true.extend(sentiment_labels_batch.cpu().numpy().tolist())
                    all_emotion_true.extend(emotion_labels_batch.cpu().numpy().tolist())

            # Sentiment Metrics
            sentiment_acc = accuracy_score(all_sentiment_true, all_sentiment_preds)
            sentiment_f1 = f1_score(all_sentiment_true, all_sentiment_preds, average="macro")
            sentiment_prec = precision_score(all_sentiment_true, all_sentiment_preds, average="macro")
            sentiment_rec = recall_score(all_sentiment_true, all_sentiment_preds, average="macro")

            # Emotion Metrics
            emotion_acc = accuracy_score(all_emotion_true, all_emotion_preds)
            emotion_f1 = f1_score(all_emotion_true, all_emotion_preds, average="macro")
            emotion_prec = precision_score(all_emotion_true, all_emotion_preds, average="macro")
            emotion_rec = recall_score(all_emotion_true, all_emotion_preds, average="macro")

            # Store metrics for each fold
            seed_metrics["sentiment_accuracy"].append(sentiment_acc)
            seed_metrics["emotion_accuracy"].append(emotion_acc)
            seed_metrics["sentiment_macro_f1"].append(sentiment_f1)
            seed_metrics["emotion_macro_f1"].append(emotion_f1)
            seed_metrics["sentiment_macro_precision"].append(sentiment_prec)
            seed_metrics["emotion_macro_precision"].append(emotion_prec)
            seed_metrics["sentiment_macro_recall"].append(sentiment_rec)
            seed_metrics["emotion_macro_recall"].append(emotion_rec)

        # After processing all folds, compute mean and std for each metric
        print(f"🌱 Seed {seed}")
        for metric, values in seed_metrics.items():
            mean_value = np.mean(values)
            std_value = np.std(values)
            print(f"  {metric.replace('_', ' ').title()}: {mean_value:.4f} ± {std_value:.4f}")
        
        run_metrics.append(seed_metrics)

    return run_metrics

# Run multi-task evaluation
multitask_cv_results = cross_validate_distilroberta_multitask(
    texts=texts,
    sentiment_labels=sent_labels,
    emotion_labels=emo_labels,
    best_params_task=best_params["multitask"],
    seeds=[42, 123, 456, 789, 999],  # Add your specific seeds
    num_folds=5,
    max_length=128
)

Using device: cuda
Index(['id', 'text_content', 'original_text', 'type', 'score', 'subjectivity',
       'sentiment', 'emotion'],
      dtype='object')

===== CROSS-VALIDATION for MULTI-TASK =====
Sentiment Samples: 95 | Emotion Samples: 95 | Folds: 5 | Seeds: [42, 123, 456, 789, 999]

🌱 Seed 42
  Fold 1/5
    Epoch 1/6 - Loss: 0.6288
    Epoch 2/6 - Loss: 0.6959
    Epoch 3/6 - Loss: 0.6671
    Epoch 4/6 - Loss: 0.6810
    Epoch 5/6 - Loss: 0.5486
    Epoch 6/6 - Loss: 0.5827
  Fold 2/5


c:\Users\hankaixin\AppData\Local\Programs\Python\Python39\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\hankaixin\AppData\Local\Programs\Python\Python39\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


    Epoch 1/6 - Loss: 0.6709
    Epoch 2/6 - Loss: 0.7310
    Epoch 3/6 - Loss: 0.6180
    Epoch 4/6 - Loss: 0.6218
    Epoch 5/6 - Loss: 0.5936
    Epoch 6/6 - Loss: 0.6518
  Fold 3/5


c:\Users\hankaixin\AppData\Local\Programs\Python\Python39\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\hankaixin\AppData\Local\Programs\Python\Python39\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


    Epoch 1/6 - Loss: 0.6139
    Epoch 2/6 - Loss: 0.7771
    Epoch 3/6 - Loss: 0.6188
    Epoch 4/6 - Loss: 0.6393
    Epoch 5/6 - Loss: 0.5786
    Epoch 6/6 - Loss: 0.6117
  Fold 4/5


...

c:\Users\hankaixin\AppData\Local\Programs\Python\Python39\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\hankaixin\AppData\Local\Programs\Python\Python39\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


    Epoch 1/6 - Loss: 0.7099
    Epoch 2/6 - Loss: 0.5508
    Epoch 3/6 - Loss: 0.5578
    Epoch 4/6 - Loss: 0.5469
    Epoch 5/6 - Loss: 0.5597
    Epoch 6/6 - Loss: 0.5697
  Fold 5/5


c:\Users\hankaixin\AppData\Local\Programs\Python\Python39\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\hankaixin\AppData\Local\Programs\Python\Python39\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


    Epoch 1/6 - Loss: 0.6716
    Epoch 2/6 - Loss: 0.5938
    Epoch 3/6 - Loss: 0.7122
    Epoch 4/6 - Loss: 0.6426
    Epoch 5/6 - Loss: 0.6411
    Epoch 6/6 - Loss: 0.6190
🌱 Seed 999
  Sentiment Accuracy: 0.5474 ± 0.1272
  Emotion Accuracy: 0.2526 ± 0.1219
  Sentiment Macro F1: 0.2326 ± 0.0388
  Emotion Macro F1: 0.1194 ± 0.0625
  Sentiment Macro Precision: 0.1825 ± 0.0424
  Emotion Macro Precision: 0.1170 ± 0.0985
  Sentiment Macro Recall: 0.3333 ± 0.0000
  Emotion Macro Recall: 0.1973 ± 0.0639


c:\Users\hankaixin\AppData\Local\Programs\Python\Python39\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\hankaixin\AppData\Local\Programs\Python\Python39\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
